In [0]:
%pip install -U mlflow transformers sentence_transformers

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.removeAll()

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.dropdown("input_table_name", input_table_list[0], input_table_list)
dbutils.widgets.dropdown("embedding_model", embedding_model_list[0], embedding_model_list)

catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
input_table_name=dbutils.widgets.get("input_table_name")
dataset_name=input_table_name
embedding_model=dbutils.widgets.get("embedding_model")

source_table = f"{catalog}.{schema}.{input_table_name}"
print(f"table_name:{source_table}, embedding_model:{embedding_model}")
catalog="amit"
#embedding_models={
#    "databricks-qwen3-embedding-0-6b": {"serving": "endpoint"},
#    "marbert-matryoshka": {"serving": "batch", "location" : f"/Volumes/{catalog}/bertopic/models/{embedding_model}"}
#}

#serving_mode=embedding_models[embedding_model]["serving_mode"]
target_table = "amit.bertopic.embeded_text"


In [0]:
df = spark.read.table(f"{catalog}.{schema}.{input_table_name}")

In [0]:
from sentence_transformers import SentenceTransformer

# Every subsequent run
model = SentenceTransformer(
    model_location_template.format(catalog=catalog, model=embedding_model)
)

In [0]:
serving_mode=embedding_model_dct[embedding_model]["serving"]

In [0]:
if serving_mode=="endpoint":
    add_column_batch(source_table, model_name=embedding_model,ai_result_column="text_embedding",input_column="text", column_type="ARRAY<FLOAT>" , is_ai=True)
elif serving_mode == "batch":
    from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType
    from sentence_transformers import SentenceTransformer

    # Every subsequent run
    model = SentenceTransformer(
        model_location_template.format(catalog=catalog, model=embedding_model)
    )
    # Config
    batch_size = 100
    truncate_dim = 256  # Optional: set to None for full 768

    # Create target table with composite key columns
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            id STRING,
            embedding_model STRING,
            dataset_name STRING,
            text_embedding ARRAY<FLOAT>,
            embedding_status STRING
        )
    """)

    def get_pending_count():
        """Count records not yet embedded for this model+dataset combination."""
        return spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {source_table} s
            WHERE s.text IS NOT NULL AND s.text != ''
            AND NOT EXISTS (
                SELECT 1 FROM {target_table} t 
                WHERE t.id = s.id 
                  AND t.embedding_model = '{embedding_model}'
                  AND t.dataset_name = '{dataset_name}'
            )
        """).collect()[0]["c"]

    def get_next_batch():
        """Get next batch — uses composite key to check what's already processed."""
        return spark.sql(f"""
            SELECT id, text
            FROM {source_table}
            WHERE text IS NOT NULL AND text != ''
            AND id NOT IN (
                SELECT id FROM {target_table} 
                WHERE embedding_model = '{embedding_model}' 
                  AND dataset_name = '{dataset_name}'
            )
            LIMIT {batch_size}
        """).toPandas()

    def embed_and_write_batch(batch_df):
        """Embed texts and write to target table with model and dataset metadata."""
        
        texts = batch_df["text"].tolist()
        embeddings = model.encode(texts, show_progress_bar=False)
        
        if truncate_dim is not None:
            embeddings = embeddings[:, :truncate_dim]
        
        # Build rows with composite key columns
        rows = []
        for i, row in batch_df.iterrows():
            rows.append((
                row["id"],
                embedding_model,
                dataset_name,
                embeddings[i].tolist(),
                "success"
            ))
        
        schema = StructType([
            StructField("id", StringType(), False),
            StructField("embedding_model", StringType(), False),
            StructField("dataset_name", StringType(), False),
            StructField("text_embedding", ArrayType(FloatType()), True),
            StructField("embedding_status", StringType(), True)
        ])
        
        result_df = spark.createDataFrame(rows, schema=schema)
        result_df.write.mode("append").saveAsTable(target_table)

    # Main loop
    batch_num = 0
    while True:
        pending = get_pending_count()
        batch_num += 1
        print(f"Remaining records to process: {pending}. Processing batch {batch_num}...")

        if pending == 0:
            print("Done. All records embedded.")
            break

        batch_df = get_next_batch()

        if batch_df.empty:
            print("No more records to process.")
            break

        start = time.perf_counter()
        embed_and_write_batch(batch_df)
        elapsed = time.perf_counter() - start
        print(f"Batch {batch_num} ({len(batch_df)} records) latency: {elapsed:.2f} sec")

    print(f"\nTotal batches processed: {batch_num - 1}")

In [0]:
#spark.sql(f"""
#    ALTER TABLE {target_table} ADD COLUMNS (
#        embedding_model STRING,
#        dataset_name STRING
#    )
#""")


In [0]:
target_table

In [0]:
dataset_name

In [0]:
#%sql
#UPDATE amit.bertopic.embeded_text
#SET embedding_model = 'databricks-qwen3-embedding-0-6b', 
#    dataset_name = 'Arabic_offensive_comment_detection_annotation_input' 
#WHERE embedding_model IS NULL;